## Milestone 2 — Single-Head Attention: Forward & Backward from Scratch

In [2]:
import numpy as np

# Set seed so results are reproducible
np.random.seed(42)

# Dummy test shapes: n=4 queries, m=5 keys/values, d=8 feature dim, dv=6 output dim
n, m, d, dv = 4, 5, 8, 6
Q = np.random.randn(n, d)
K = np.random.randn(m, d)
V = np.random.randn(m, dv)
dO = np.random.randn(n, dv)  # Upstream gradient coming from the loss


**Forward and Backward Pass Implementation:**

In [4]:
def attention_forward(Q, K, V):
    # Scale factor 1/sqrt(d) keeps variance at 1 so softmax doesn't saturate
    scale = 1.0 / np.sqrt(Q.shape[-1])
    
    # Raw dot products: shape (n, m)
    S = (Q @ K.T) * scale
    
    # Numerical stability fix: subtract max in each row so exp() doesn't explode to inf
    S_max = np.max(S, axis=-1, keepdims=True)
    exp_S = np.exp(S - S_max)
    
    # Row-wise softmax probabilities: shape (n, m)
    P = exp_S / np.sum(exp_S, axis=-1, keepdims=True)
    
    # Weighted sum of values: shape (n, dv)
    O = P @ V
    
    cache = (Q, K, V, S, P, scale)
    return O, cache


def attention_backward(dO, cache):
    Q, K, V, S, P, scale = cache
    
    # 1. Gradient w.r.t V: O = P @ V  -->  dV = P.T @ dO
    dV = P.T @ dO
    
    # 2. Gradient w.r.t P: O = P @ V  -->  dP = dO @ V.T
    dP = dO @ V.T
    
    # 3. Softmax gradient w.r.t S:
    # dS_ij = P_ij * (dP_ij - sum_k(dP_ik * P_ik))
    # Neat vectorized trick: sum(dP * P) is the expected gradient per row
    dP_dot_P = np.sum(dP * P, axis=-1, keepdims=True)
    dS = P * (dP - dP_dot_P)
    
    # 4. Gradients w.r.t Q and K through S = (Q @ K.T) / sqrt(d)
    dQ = (dS @ K) * scale
    dK = (dS.T @ Q) * scale
    
    return dQ, dK, dV, dP, dS


**Numerical Gradient Checking:**

In [6]:
# Checking analytical gradients against two-sided finite difference:
# Formula: grad_num = (loss(x + h) - loss(x - h)) / (2 * h)
# Step size: h = 1e-5 (the sweet spot: 1e-7 gets hit by float cancellation error, 1e-3 has truncation error)
# Precision: float64
# Acceptance threshold: 1e-7

h = 1e-5
threshold = 1e-7

def rel_err(a, b):
    # Relative error formula to prevent division by zero
    return np.max(np.abs(a - b) / np.maximum(1e-15, np.abs(a) + np.abs(b)))

# Run analytical pass
O, cache = attention_forward(Q, K, V)
_, _, _, S, P, _ = cache
dQ, dK, dV, dP, dS = attention_backward(dO, cache)

# 1. Numerical check for V
num_dV = np.zeros_like(V)
for i in range(V.shape[0]):
    for j in range(V.shape[1]):
        V_pos = V.copy(); V_pos[i, j] += h
        V_neg = V.copy(); V_neg[i, j] -= h
        O_pos, _ = attention_forward(Q, K, V_pos)
        O_neg, _ = attention_forward(Q, K, V_neg)
        num_dV[i, j] = np.sum((O_pos - O_neg) * dO) / (2 * h)

# 2. Numerical check for P
num_dP = np.zeros_like(P)
for i in range(P.shape[0]):
    for j in range(P.shape[1]):
        P_pos = P.copy(); P_pos[i, j] += h
        P_neg = P.copy(); P_neg[i, j] -= h
        num_dP[i, j] = np.sum(((P_pos @ V) - (P_neg @ V)) * dO) / (2 * h)

# 3. Numerical check for S
num_dS = np.zeros_like(S)
for i in range(S.shape[0]):
    for j in range(S.shape[1]):
        S_pos = S.copy(); S_pos[i, j] += h
        S_neg = S.copy(); S_neg[i, j] -= h
        P_pos = np.exp(S_pos - np.max(S_pos, axis=-1, keepdims=True))
        P_pos /= np.sum(P_pos, axis=-1, keepdims=True)
        P_neg = np.exp(S_neg - np.max(S_neg, axis=-1, keepdims=True))
        P_neg /= np.sum(P_neg, axis=-1, keepdims=True)
        num_dS[i, j] = np.sum(((P_pos @ V) - (P_neg @ V)) * dO) / (2 * h)

# 4. Numerical check for Q
num_dQ = np.zeros_like(Q)
for i in range(Q.shape[0]):
    for j in range(Q.shape[1]):
        Q_pos = Q.copy(); Q_pos[i, j] += h
        Q_neg = Q.copy(); Q_neg[i, j] -= h
        O_pos, _ = attention_forward(Q_pos, K, V)
        O_neg, _ = attention_forward(Q_neg, K, V)
        num_dQ[i, j] = np.sum((O_pos - O_neg) * dO) / (2 * h)

# 5. Numerical check for K
num_dK = np.zeros_like(K)
for i in range(K.shape[0]):
    for j in range(K.shape[1]):
        K_pos = K.copy(); K_pos[i, j] += h
        K_neg = K.copy(); K_neg[i, j] -= h
        O_pos, _ = attention_forward(Q, K_pos, V)
        O_neg, _ = attention_forward(Q, K_neg, V)
        num_dK[i, j] = np.sum((O_pos - O_neg) * dO) / (2 * h)

print(f"dV error: {rel_err(dV, num_dV):.4e} -> {'PASS' if rel_err(dV, num_dV) < threshold else 'FAIL'}")
print(f"dP error: {rel_err(dP, num_dP):.4e} -> {'PASS' if rel_err(dP, num_dP) < threshold else 'FAIL'}")
print(f"dS error: {rel_err(dS, num_dS):.4e} -> {'PASS' if rel_err(dS, num_dS) < threshold else 'FAIL'}")
print(f"dQ error: {rel_err(dQ, num_dQ):.4e} -> {'PASS' if rel_err(dQ, num_dQ) < threshold else 'FAIL'}")
print(f"dK error: {rel_err(dK, num_dK):.4e} -> {'PASS' if rel_err(dK, num_dK) < threshold else 'FAIL'}")


dV error: 2.4133e-11 -> PASS
dP error: 1.1954e-11 -> PASS
dS error: 5.2062e-10 -> PASS
dQ error: 3.5720e-10 -> PASS
dK error: 4.7704e-09 -> PASS


**Causal Masking in Attention (Forward & Backward):**

In [8]:
# For causal attention (like GPT), tokens can't look ahead at future tokens.
# We set future entries (j > i) to -1e9 so their softmax weight becomes 0.
# In backward pass, their gradient must also be explicitly set to 0.

def causal_attention_forward(Q, K, V):
    scale = 1.0 / np.sqrt(Q.shape[-1])
    S = (Q @ K.T) * scale
    
    # Mask out upper triangle (future positions j > i)
    mask = np.triu(np.ones_like(S), k=1).astype(bool)
    S_masked = np.where(mask, -1e9, S)
    
    S_max = np.max(S_masked, axis=-1, keepdims=True)
    exp_S = np.exp(S_masked - S_max)
    exp_S = np.where(mask, 0.0, exp_S) # Force exact zero on masked positions
    P = exp_S / np.sum(exp_S, axis=-1, keepdims=True)
    O = P @ V
    return O, (Q, K, V, S, P, mask, scale)

def causal_attention_backward(dO, cache):
    Q, K, V, S, P, mask, scale = cache
    dV = P.T @ dO
    dP = dO @ V.T
    
    dS = P * (dP - np.sum(dP * P, axis=-1, keepdims=True))
    # Key fix: masked positions contributed 0 to forward, so their gradient must be 0
    dS = np.where(mask, 0.0, dS)
    
    dQ = (dS @ K) * scale
    dK = (dS.T @ Q) * scale
    return dQ, dK, dV

O_c, cache_c = causal_attention_forward(Q, K, V)
dQ_c, dK_c, dV_c = causal_attention_backward(dO, cache_c)

# Numerical check on causal Q
num_dQ_c = np.zeros_like(Q)
for i in range(Q.shape[0]):
    for j in range(Q.shape[1]):
        Q_pos = Q.copy(); Q_pos[i, j] += h
        Q_neg = Q.copy(); Q_neg[i, j] -= h
        O_pos, _ = causal_attention_forward(Q_pos, K, V)
        O_neg, _ = causal_attention_forward(Q_neg, K, V)
        num_dQ_c[i, j] = np.sum((O_pos - O_neg) * dO) / (2 * h)

print(f"Causal dQ error: {rel_err(dQ_c, num_dQ_c):.4e} -> {'PASS' if rel_err(dQ_c, num_dQ_c) < threshold else 'FAIL'}")


Causal dQ error: 2.3517e-09 -> PASS
